# Solrへの登録のシンプルなサンプル

Solrをベクトルデータベースとして使用するサンプル。

※Solrのバージョン：9.8.1

Solr 9.0以降でベクトル検索機能が追加され、dense_vector型を使用してベクトル検索が可能になりました。

## 必要パッケージのインポート

In [ ]:
import pysolr
import requests
from sentence_transformers import SentenceTransformer
import time
import json

## 設定

In [ ]:
SOLR_URL = 'http://llm-rag-examples-solr:8983/solr'
COLLECTION_NAME = 'vector_test01'

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
MODEL_DIM = 384

## 埋め込みモデル初期化

In [ ]:
model = SentenceTransformer(MODEL_NAME)

## Solrコレクションの設定

### コレクションの有無チェック、あれば削除

In [ ]:
# コレクション一覧取得
try:
    response = requests.get(f'{SOLR_URL}/admin/collections?action=LIST&wt=json')
    collections = response.json().get('collections', [])
    
    if COLLECTION_NAME in collections:
        # コレクション削除
        delete_response = requests.get(f'{SOLR_URL}/admin/collections?action=DELETE&name={COLLECTION_NAME}&wt=json')
        print(f'Collection {COLLECTION_NAME} is deleted.')
        time.sleep(2)  # 削除完了まで少し待機
    else:
        print(f'Collection {COLLECTION_NAME} does not exist.')
except Exception as e:
    print(f'Error checking collections: {e}')

### コレクション作成

In [ ]:
# コレクション作成
create_params = {
    'action': 'CREATE',
    'name': COLLECTION_NAME,
    'numShards': 1,
    'replicationFactor': 1,
    'wt': 'json'
}

create_response = requests.get(f'{SOLR_URL}/admin/collections', params=create_params)
print(f"Collection '{COLLECTION_NAME}' creation response: {create_response.json()}")

# 作成完了まで少し待機
time.sleep(3)

### スキーマ設定（フィールド追加）

In [ ]:
# テキストフィールドの追加
text_field = {
    "add-field": {
        "name": "text",
        "type": "text_general",
        "stored": True,
        "indexed": True
    }
}

text_response = requests.post(
    f'{SOLR_URL}/{COLLECTION_NAME}/schema',
    json=text_field,
    headers={'Content-Type': 'application/json'}
)
print(f"Text field addition response: {text_response.json()}")

In [ ]:
# ベクトルフィールドの追加（dense_vector型）
vector_field = {
    "add-field": {
        "name": "vector",
        "type": "knn_vector",
        "stored": True,
        "indexed": True,
        "dimension": MODEL_DIM,
        "similarityFunction": "cosine"
    }
}

vector_response = requests.post(
    f'{SOLR_URL}/{COLLECTION_NAME}/schema',
    json=vector_field,
    headers={'Content-Type': 'application/json'}
)
print(f"Vector field addition response: {vector_response.json()}")

## Solrクライアント接続

In [ ]:
solr = pysolr.Solr(f'{SOLR_URL}/{COLLECTION_NAME}', always_commit=True)

## ドキュメントをインデックス

In [ ]:
# --- 登録するテキストデータ ---
texts = [
    '猫は可愛い動物です。',
    '犬は人間の親友と呼ばれています。',
    '東京は日本の首都です。'
]

In [ ]:
documents = []
for i, text in enumerate(texts):
    vector = model.encode(text)
    doc = {
        'id': str(i),
        'text': text,
        'vector': vector.tolist()
    }
    documents.append(doc)
    print(f"Prepared: {text}")

# 一括登録
solr.add(documents)
print(f"Indexed {len(documents)} documents to Solr collection '{COLLECTION_NAME}'")

## データ登録確認

In [ ]:
# 全ドキュメント取得
results = solr.search('*:*')
print(f"Total documents: {results.hits}")
for doc in results:
    print(f"ID: {doc['id']}, Text: {doc['text']}")